# Fine-Tuning VadCLIP Có Hướng Dẫn Bằng Description Trên Colab

Notebook này chỉ đóng vai trò runner để chạy các file Python trong `VadCLIP/src`. Source code model vẫn nằm trong thư mục `VadCLIP/src`, không được copy trực tiếp vào notebook.


## 1. Mount Google Drive Và Cấu Hình Đường Dẫn

Cell này mount Google Drive, khai báo thư mục dự án, thư mục feature, checkpoint baseline và các đường dẫn dùng chung cho những cell phía sau.


In [ ]:
from pathlib import Path
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Not running in Colab or Drive is already mounted.')

PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
DRIVE_FEATURE_ROOT = PROJECT_ROOT / 'UCFClipFeatures'
DRIVE_FEATURE_ARCHIVES = [
    PROJECT_ROOT / 'UCFClipFeatures.tar',
    PROJECT_ROOT / 'UCFClipFeatures.tar.gz',
    PROJECT_ROOT / 'UCFClipFeatures.tgz',
    PROJECT_ROOT / 'UCFClipFeatures.zip',
]
LOCAL_FEATURE_ROOT = Path('/content/UCFClipFeatures')
FEATURE_ROOT = DRIVE_FEATURE_ROOT
DESCRIPTION_JSON = PROJECT_ROOT / 'code' / 'ucf_gpt_video_descriptions.json'
USE_VADCLIP_CHECKPOINT = False
PRETRAINED_MODEL = PROJECT_ROOT / 'model_ucf.pth'

sys.path.insert(0, str(SRC_DIR))
os.chdir(SRC_DIR)

print('Project root:', PROJECT_ROOT)
print('Source dir:', SRC_DIR)
print('Drive feature root:', DRIVE_FEATURE_ROOT)
print('Feature root:', FEATURE_ROOT)
print('Use VadCLIP checkpoint:', USE_VADCLIP_CHECKPOINT)

## 2. Cài Đặt Dependencies

Cell này cài các thư viện cần thiết cho VadCLIP trên Colab, bao gồm CLIP tokenizer, sklearn, scipy và matplotlib.


In [ ]:
!pip -q install ftfy regex tqdm scikit-learn scipy matplotlib


## 3. Kiểm Tra Các File Bắt Buộc

Cell này kiểm tra các file/folder quan trọng trước khi train, ví dụ source code, feature root, list train/test, ground truth và checkpoint baseline.


In [ ]:
required_paths = [
    SRC_DIR / 'model_description.py',
    SRC_DIR / 'ucf_train_description.py',
    SRC_DIR / 'ucf_train_class_semantic.py',
    SRC_DIR / 'ucf_option_class_semantic.py',
    SRC_DIR / 'ucf_train_class_prototype.py',
    SRC_DIR / 'ucf_option_class_prototype.py',
    SRC_DIR / 'model_prompt_replacement.py',
    SRC_DIR / 'ucf_train_prompt_replacement.py',
    SRC_DIR / 'ucf_option_prompt_replacement.py',
    SRC_DIR / 'model_multi_prompt.py',
    SRC_DIR / 'ucf_train_multi_prompt.py',
    SRC_DIR / 'ucf_option_multi_prompt.py',
    SRC_DIR / 'ucf_evaluate.py',
    SRC_DIR / 'ucf_analyze_checkpoints.py',
    SRC_DIR / 'ucf_analyze_manual_prompts.py',
    SRC_DIR / 'utils' / 'dataset_description.py',
    SRC_DIR / 'utils' / 'prompt_replacement.py',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgb_description.csv',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgbtest_description.csv',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgbtest_relative.csv',
    PRETRAINED_MODEL,
    DESCRIPTION_JSON,
    PROJECT_ROOT / 'code' / 'ucf_class_description_prototypes_vadclip_train.json',
    PROJECT_ROOT / 'code' / 'ucf_compact_class_prototypes.json',
    PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_10x.json',
    PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_10x_v2.json',
    PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_5x_v3.json',
    PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_5x_v4.json',
]
if USE_VADCLIP_CHECKPOINT:
    required_paths.append(PRETRAINED_MODEL)

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('\n'.join(missing))

available_feature_archive = next((path for path in DRIVE_FEATURE_ARCHIVES if path.exists()), None)
if not DRIVE_FEATURE_ROOT.exists() and available_feature_archive is None:
    archive_names = ', '.join(path.name for path in DRIVE_FEATURE_ARCHIVES)
    raise FileNotFoundError(f'Missing UCFClipFeatures folder or one archive: {archive_names}')

print('All required files exist.')

## 4. Copy Feature Sang Runtime Local

Cell này copy `UCFClipFeatures` từ Google Drive sang `/content` để giảm I/O khi train. Nên chạy cell này nếu bạn muốn train nhanh hơn trên Colab.


In [ ]:
import shutil
import subprocess
import time

subprocess.run(['df', '-h', '/content'], check=False)
available_feature_archive = next((path for path in DRIVE_FEATURE_ARCHIVES if path.exists()), None)
if available_feature_archive is not None:
    subprocess.run(['du', '-sh', str(available_feature_archive)], check=False)
else:
    subprocess.run(['du', '-sh', str(DRIVE_FEATURE_ROOT)], check=False)

start = time.time()
if available_feature_archive is not None:
    local_archive = Path('/content') / available_feature_archive.name
    if not local_archive.exists() or local_archive.stat().st_size != available_feature_archive.stat().st_size:
        print('Copying archive to local runtime:', available_feature_archive)
        shutil.copy2(available_feature_archive, local_archive)
    else:
        print('Local archive already exists with matching size:', local_archive)

    print('Extracting archive:', local_archive)
    if local_archive.suffix == '.zip':
        subprocess.run(['unzip', '-q', '-o', str(local_archive), '-d', '/content'], check=True)
    else:
        subprocess.run(['tar', '-xf', str(local_archive), '-C', '/content'], check=True)

    if not LOCAL_FEATURE_ROOT.exists():
        raise FileNotFoundError('Archive must contain top-level folder UCFClipFeatures/')
else:
    LOCAL_FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
    if shutil.which('rsync'):
        subprocess.run([
            'rsync', '-ah', '--info=progress2',
            f'{DRIVE_FEATURE_ROOT}/', f'{LOCAL_FEATURE_ROOT}/'
        ], check=True)
    else:
        subprocess.run(['cp', '-r', f'{DRIVE_FEATURE_ROOT}/.', str(LOCAL_FEATURE_ROOT)], check=True)

FEATURE_ROOT = LOCAL_FEATURE_ROOT
print(f'Prepared local features in {(time.time() - start) / 60:.1f} minutes')
print('Feature root:', FEATURE_ROOT)


## 5. Kiểm Tra Độ Phủ Feature

Cell này kiểm tra các file `.npy` được list train/test tham chiếu có tồn tại trong `FEATURE_ROOT` hay không. Nên chạy trước khi train để tránh lỗi thiếu feature giữa chừng.


In [ ]:
import csv
from collections import Counter

def check_feature_coverage(csv_path, feature_root, preview=30):
    rows = list(csv.DictReader(open(csv_path, encoding='utf-8')))
    missing = []
    for row in rows:
        path = feature_root / row['path']
        if not path.exists():
            missing.append(row)

    print(f'{csv_path.name}: rows={len(rows)}, missing_files={len(missing)}')
    if missing:
        print('Missing by label:', dict(Counter(row['label'] for row in missing)))
        for row in missing[:preview]:
            print(f"  {row.get('video_id', '')},{row['label']},{row['path']}")
        if len(missing) > preview:
            print(f'  ... and {len(missing) - preview} more')
        raise FileNotFoundError(f'{csv_path.name} references missing feature files. Re-upload UCFClipFeatures or rebuild the list from available files.')

for csv_path in [
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgb_description.csv',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgbtest_description.csv',
    PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgbtest_relative.csv',
]:
    check_feature_coverage(csv_path, FEATURE_ROOT)


## 6. Smoke Test Dataset

Cell này thử load một vài sample từ dataset để kiểm tra shape feature, label, video id và đường dẫn feature có đúng không.


In [ ]:
from utils.dataset_description import UCFDescriptionDataset

label_map = {
    'Normal': 'normal', 'Abuse': 'abuse', 'Arrest': 'arrest', 'Arson': 'arson',
    'Assault': 'assault', 'Burglary': 'burglary', 'Explosion': 'explosion',
    'Fighting': 'fighting', 'RoadAccidents': 'roadAccidents', 'Robbery': 'robbery',
    'Shooting': 'shooting', 'Shoplifting': 'shoplifting', 'Stealing': 'stealing',
    'Vandalism': 'vandalism'
}

train_list = PROJECT_ROOT / 'VadCLIP' / 'list' / 'ucf_CLIP_rgb_description.csv'
dataset = UCFDescriptionDataset(256, str(train_list), False, label_map, str(FEATURE_ROOT), str(DESCRIPTION_JSON), normal=False)
feature, label, length, video_id, description = dataset[0]
print('Dataset size:', len(dataset))
print('Feature shape:', feature.shape)
print('Label:', label)
print('Length:', length)
print('Video ID:', video_id)
print('Description:', description[:200])

## 7. Fine-Tune Description-Guided VadCLIP

Cell này là pipeline fine-tune cũ dùng video description. Hiện tại chủ yếu giữ lại để tham chiếu lịch sử thí nghiệm; các hướng prompt/prototype mới nằm ở các section phía dưới.


In [ ]:
import shlex
import subprocess

def run_command(cmd):
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

train_description_cmd = [
    'python', 'ucf_train_description.py',
    '--feature-root', str(FEATURE_ROOT),
    '--description-json', str(DESCRIPTION_JSON),
    '--use-pretrained-model', str(USE_VADCLIP_CHECKPOINT).lower(),
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lambda-desc', '0.003',
    '--lambda-contrastive', '0.003',
    '--lambda-distill', '0.05',
    '--desc-loss-type', 'contrastive',
    '--desc-pooling', 'topk',
    '--top-k-ratio', '0.15',
    '--contrastive-temperature', '0.07',
    '--contrastive-samples', 'all',
    '--use-desc-projection', 'true',
    '--distill-baseline', str(USE_VADCLIP_CHECKPOINT).lower(),
    '--trainable-scope', 'all',
    '--lr', '1e-5',
    '--max-epoch', '3',
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--cache-description-embeddings', 'true',
    '--epoch-checkpoint-dir', 'model/epoch_checkpoints_contrastive',
    '--output-model-path', 'model/model_ucf_description_contrastive.pth',
]
if USE_VADCLIP_CHECKPOINT:
    train_description_cmd.extend(['--pretrained-model-path', str(PRETRAINED_MODEL)])

print('Running:', ' '.join(shlex.quote(part) for part in train_description_cmd))
subprocess.run(train_description_cmd, check=True)

## 8. Đánh Giá Baseline Và Model Fine-Tuned

Cell này chạy evaluator để so sánh checkpoint baseline với model fine-tuned theo các metric của VadCLIP: AUC, AP, Ano-AUC và mAP.


In [ ]:
!python ucf_evaluate.py \
  --feature-root "{FEATURE_ROOT}" \
  --test-list "../list/ucf_CLIP_rgbtest_relative.csv" \
  --baseline-model-path "{PRETRAINED_MODEL}" \
  --description-model-path "model/model_ucf_description_contrastive.pth" \
  --gt-path "../list/gt_ucf.npy" \
  --gt-segment-path "../list/gt_segment_ucf.npy" \
  --gt-label-path "../list/gt_label_ucf.npy"


## 9. Phân Tích Checkpoint Và Vẽ Diagnostics

Cell này chạy script phân tích checkpoint theo epoch, lưu bảng metric, biểu đồ score distribution, per-class drift và timeline visualization.


In [ ]:
!python ucf_analyze_checkpoints.py \
  --feature-root "{FEATURE_ROOT}" \
  --test-list "../list/ucf_CLIP_rgbtest_relative.csv" \
  --baseline-model-path "{PRETRAINED_MODEL}" \
  --epoch-checkpoint-dir "model/epoch_checkpoints_contrastive" \
  --description-model-path "model/model_ucf_description_contrastive.pth" \
  --output-dir "{PROJECT_ROOT / 'code' / 'ucf_checkpoint_diagnostics_contrastive'}" \
  --gt-path "../list/gt_ucf.npy" \
  --gt-segment-path "../list/gt_segment_ucf.npy" \
  --gt-label-path "../list/gt_label_ucf.npy" \
  --timeline-count 6


## 10. Fine-Tune Theo Class Semantic

Cell này chạy hướng class semantic regularization: dùng description để tạo soft target theo class, sau đó regularize phân phối class từ `logits2`. Hướng này được giữ lại để đối chiếu kết quả.


In [ ]:
train_class_semantic_cmd = [
    'python', 'ucf_train_class_semantic.py',
    '--feature-root', str(FEATURE_ROOT),
    '--description-json', str(DESCRIPTION_JSON),
    '--use-pretrained-model', str(USE_VADCLIP_CHECKPOINT).lower(),
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lambda-sem', '0.05',
    '--semantic-alpha', '0.2',
    '--semantic-temperature', '0.07',
    '--video-logit-pooling', 'topk',
    '--top-k-ratio', '0.15',
    '--lr', '1e-5',
    '--max-epoch', '3',
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--cache-semantic-targets', 'true',
    '--epoch-checkpoint-dir', 'model/epoch_checkpoints_class_semantic',
    '--output-model-path', 'model/model_ucf_class_semantic.pth',
]
if USE_VADCLIP_CHECKPOINT:
    train_class_semantic_cmd.extend(['--pretrained-model-path', str(PRETRAINED_MODEL)])

print('Running:', ' '.join(shlex.quote(part) for part in train_class_semantic_cmd))
subprocess.run(train_class_semantic_cmd, check=True)


## 11. Đánh Giá Model Class Semantic

Cell này đánh giá model class semantic bằng evaluator chung, inference vẫn không dùng description.


In [ ]:
!python ucf_evaluate.py \
  --feature-root "{FEATURE_ROOT}" \
  --test-list "../list/ucf_CLIP_rgbtest_relative.csv" \
  --baseline-model-path "{PRETRAINED_MODEL}" \
  --description-model-path "model/model_ucf_class_semantic.pth" \
  --description-model-type baseline \
  --gt-path "../list/gt_ucf.npy" \
  --gt-segment-path "../list/gt_segment_ucf.npy" \
  --gt-label-path "../list/gt_label_ucf.npy"


## 12. Phân Tích Checkpoint Class Semantic

Cell này tạo diagnostics cho các checkpoint của hướng class semantic để xem metric thay đổi theo epoch và class nào bị lệch.


In [ ]:
!python ucf_analyze_checkpoints.py \
  --feature-root "{FEATURE_ROOT}" \
  --test-list "../list/ucf_CLIP_rgbtest_relative.csv" \
  --baseline-model-path "{PRETRAINED_MODEL}" \
  --epoch-checkpoint-dir "model/epoch_checkpoints_class_semantic" \
  --description-model-path "model/model_ucf_class_semantic.pth" \
  --finetuned-model-type baseline \
  --output-dir "{PROJECT_ROOT / 'code' / 'ucf_checkpoint_diagnostics_class_semantic'}" \
  --gt-path "../list/gt_ucf.npy" \
  --gt-segment-path "../list/gt_segment_ucf.npy" \
  --gt-label-path "../list/gt_label_ucf.npy" \
  --timeline-count 6


## 13. Fine-Tune Class Prototype Từ Scratch

Cell này chạy hướng class prototype branch từ scratch. Model không load `model_ucf.pth` để khởi tạo training; checkpoint baseline chỉ dùng cho đánh giá/so sánh sau train.


In [ ]:
import shlex
import subprocess

train_class_prototype_cmd = [
    'python', 'ucf_train_class_prototype.py',
    '--feature-root', str(FEATURE_ROOT),
    '--prototype-json', str(PROJECT_ROOT / 'code' / 'ucf_class_description_prototypes_vadclip_train.json'),
    '--use-pretrained-model', str(USE_VADCLIP_CHECKPOINT).lower(),
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lambda-proto-mil', '1.0',
    '--lambda-proto', '0.05',
    '--prototype-temperature', '0.07',
    '--prototype-mode', 'centroid',
    '--lr', '2e-5',
    '--max-epoch', '10',
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--epoch-checkpoint-dir', 'model/epoch_checkpoints_class_prototype_scratch',
    '--output-model-path', 'model/model_ucf_class_prototype_scratch.pth',
]
if USE_VADCLIP_CHECKPOINT:
    train_class_prototype_cmd.extend(['--pretrained-model-path', str(PRETRAINED_MODEL)])

print('Running:', ' '.join(shlex.quote(part) for part in train_class_prototype_cmd))
subprocess.run(train_class_prototype_cmd, check=True)


## 14. Đánh Giá Model Class Prototype

Cell này đánh giá model class prototype sau khi train, so sánh trực tiếp với baseline VadCLIP.


In [ ]:
!python ucf_evaluate.py \
  --feature-root "{FEATURE_ROOT}" \
  --test-list "../list/ucf_CLIP_rgbtest_relative.csv" \
  --baseline-model-path "{PRETRAINED_MODEL}" \
  --description-model-path "model/model_ucf_class_prototype_scratch.pth" \
  --description-model-type baseline \
  --gt-path "../list/gt_ucf.npy" \
  --gt-segment-path "../list/gt_segment_ucf.npy" \
  --gt-label-path "../list/gt_label_ucf.npy"


## 15. Phân Tích Checkpoint Class Prototype

Cell này phân tích các checkpoint của hướng class prototype, bao gồm AUC, AP, Ano-AUC, mAP và per-class drift.


In [ ]:
!python ucf_analyze_checkpoints.py \
  --feature-root "{FEATURE_ROOT}" \
  --test-list "../list/ucf_CLIP_rgbtest_relative.csv" \
  --baseline-model-path "{PRETRAINED_MODEL}" \
  --epoch-checkpoint-dir "model/epoch_checkpoints_class_prototype_scratch" \
  --description-model-path "model/model_ucf_class_prototype_scratch.pth" \
  --finetuned-model-type baseline \
  --output-dir "{PROJECT_ROOT / 'code' / 'ucf_checkpoint_diagnostics_class_prototype_scratch'}" \
  --gt-path "../list/gt_ucf.npy" \
  --gt-segment-path "../list/gt_segment_ucf.npy" \
  --gt-label-path "../list/gt_label_ucf.npy" \
  --timeline-count 6


## 16. Ablation Compact Prototype

Section này thử các compact prototype ngắn từ `ucf_compact_class_prototypes.json`. Đây là nhóm thí nghiệm cũ để kiểm tra prototype ngắn có ít gây nhiễu hơn prototype dài hay không.


In [ ]:
import shlex
import subprocess

COMPACT_PROTOTYPE_JSON = PROJECT_ROOT / 'code' / 'ucf_compact_class_prototypes.json'
compact_variants = {
    'keyword': ['keyword_phrase'],
    'action': ['compact_action_phrase'],
    'sentence': ['short_visual_sentence'],
    'all': ['keyword_phrase', 'compact_action_phrase', 'short_visual_sentence'],
}

for variant_name, prototype_types in compact_variants.items():
    print('\n' + '=' * 80)
    print('Compact prototype variant:', variant_name, prototype_types)

    epoch_dir = f'model/epoch_checkpoints_compact_{variant_name}'
    output_model = f'model/model_ucf_compact_{variant_name}.pth'
    checkpoint_path = f'model/checkpoint_compact_{variant_name}.pth'
    save_cur_path = f'model/model_cur_compact_{variant_name}.pth'
    diagnostics_dir = PROJECT_ROOT / 'code' / f'ucf_checkpoint_diagnostics_compact_{variant_name}'

    train_cmd = [
        'python', 'ucf_train_class_prototype.py',
        '--feature-root', str(FEATURE_ROOT),
        '--prototype-json', str(COMPACT_PROTOTYPE_JSON),
        '--prototype-schema', 'compact',
        '--compact-prototype-types', *prototype_types,
        '--use-pretrained-model', 'true',
        '--pretrained-model-path', str(PRETRAINED_MODEL),
        '--train-list', '../list/ucf_CLIP_rgb_description.csv',
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
        '--lambda-proto-mil', '0.3',
        '--lambda-proto', '0.01',
        '--prototype-temperature', '0.07',
        '--prototype-mode', 'centroid',
        '--lr', '1e-5',
        '--max-epoch', '3',
        '--eval-steps', '0',
        '--num-workers', '4',
        '--pin-memory', 'true',
        '--use-amp', 'false',
        '--epoch-checkpoint-dir', epoch_dir,
        '--output-model-path', output_model,
        '--checkpoint-path', checkpoint_path,
        '--save-cur-path', save_cur_path,
    ]

    print('Training:', ' '.join(shlex.quote(part) for part in train_cmd))
    run_command(train_cmd)

    eval_cmd = [
        'python', 'ucf_evaluate.py',
        '--feature-root', str(FEATURE_ROOT),
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--baseline-model-path', str(PRETRAINED_MODEL),
        '--description-model-path', output_model,
        '--description-model-type', 'baseline',
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
    ]
    print('Evaluating:', ' '.join(shlex.quote(part) for part in eval_cmd))
    run_command(eval_cmd)

    analyze_cmd = [
        'python', 'ucf_analyze_checkpoints.py',
        '--feature-root', str(FEATURE_ROOT),
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--baseline-model-path', str(PRETRAINED_MODEL),
        '--epoch-checkpoint-dir', epoch_dir,
        '--description-model-path', output_model,
        '--finetuned-model-type', 'baseline',
        '--output-dir', str(diagnostics_dir),
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
        '--timeline-count', '6',
    ]
    print('Analyzing:', ' '.join(shlex.quote(part) for part in analyze_cmd))
    run_command(analyze_cmd)


## 17. Thí Nghiệm Prototype Prompt Replacement

Section này thử thay class prompt bằng prototype prompt. CLIP vẫn pretrained/frozen; phần VadCLIP-specific được train từ scratch trong các cấu hình hiện tại.


In [ ]:
import shlex
import subprocess

PROMPT_REPLACEMENT_PROTOTYPE_JSON = PROJECT_ROOT / 'code' / 'ucf_compact_class_prototypes.json'
prompt_replacement_variants = {
    'prototype_only': 'prototype_only',
    'class_plus': 'class_plus_prototype',
}

for variant_name, replacement_mode in prompt_replacement_variants.items():
    print('\n' + '=' * 80)
    print('Prompt replacement variant:', variant_name, replacement_mode)

    epoch_dir = f'model/epoch_checkpoints_prompt_replace_{variant_name}_scratch'
    output_model = f'model/model_ucf_prompt_replace_{variant_name}_scratch.pth'
    checkpoint_path = f'model/checkpoint_prompt_replace_{variant_name}_scratch.pth'
    save_cur_path = f'model/model_cur_prompt_replace_{variant_name}_scratch.pth'
    diagnostics_dir = PROJECT_ROOT / 'code' / f'ucf_checkpoint_diagnostics_prompt_replace_{variant_name}_scratch'

    train_cmd = [
        'python', 'ucf_train_prompt_replacement.py',
        '--feature-root', str(FEATURE_ROOT),
        '--prototype-json', str(PROMPT_REPLACEMENT_PROTOTYPE_JSON),
        '--prompt-replacement-mode', replacement_mode,
        '--compact-prototype-types', 'keyword_phrase', 'compact_action_phrase', 'short_visual_sentence',
        '--use-pretrained-model', 'false',
        '--train-list', '../list/ucf_CLIP_rgb_description.csv',
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
        '--lr', '2e-5',
        '--max-epoch', '10',
        '--scheduler-milestones', '4', '8',
        '--eval-steps', '0',
        '--num-workers', '4',
        '--pin-memory', 'true',
        '--use-amp', 'false',
        '--epoch-checkpoint-dir', epoch_dir,
        '--output-model-path', output_model,
        '--checkpoint-path', checkpoint_path,
        '--save-cur-path', save_cur_path,
    ]
    print('Training:', ' '.join(shlex.quote(part) for part in train_cmd))
    subprocess.run(train_cmd, check=True)

    eval_cmd = [
        'python', 'ucf_evaluate.py',
        '--feature-root', str(FEATURE_ROOT),
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--baseline-model-path', str(PRETRAINED_MODEL),
        '--description-model-path', output_model,
        '--description-model-type', 'prompt_replacement',
        '--prompt-replacement-mode', replacement_mode,
        '--prototype-json', str(PROMPT_REPLACEMENT_PROTOTYPE_JSON),
        '--compact-prototype-types', 'keyword_phrase', 'compact_action_phrase', 'short_visual_sentence',
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
    ]
    print('Evaluating:', ' '.join(shlex.quote(part) for part in eval_cmd))
    subprocess.run(eval_cmd, check=True)

    analyze_cmd = [
        'python', 'ucf_analyze_checkpoints.py',
        '--feature-root', str(FEATURE_ROOT),
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--baseline-model-path', str(PRETRAINED_MODEL),
        '--epoch-checkpoint-dir', epoch_dir,
        '--description-model-path', output_model,
        '--finetuned-model-type', 'prompt_replacement',
        '--prompt-replacement-mode', replacement_mode,
        '--prototype-json', str(PROMPT_REPLACEMENT_PROTOTYPE_JSON),
        '--compact-prototype-types', 'keyword_phrase', 'compact_action_phrase', 'short_visual_sentence',
        '--output-dir', str(diagnostics_dir),
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
        '--timeline-count', '6',
    ]
    print('Analyzing:', ' '.join(shlex.quote(part) for part in analyze_cmd))
    subprocess.run(analyze_cmd, check=True)


## 18. Multi-Prompt Training Từ Một File Prompt

Cell này train model bằng một file prompt cụ thể ở chế độ `prototype_only`, tức là prompt được đưa trực tiếp vào text encoder và không tự thêm tên class phía trước. Mỗi class có nhiều prompt, model tạo alignment map `[B,T,14,K]`, sau đó aggregate thành `logits2 [B,T,14]`.

Muốn đổi file prompt thì sửa `multi_prompt_variants` và `selected_multi_prompt_variants` trong cell bên dưới. Các regularizer chống tăng score negative cũng đã được thêm vào cell này và có thể bật bằng cách đổi lambda.


In [ ]:
import shlex
import subprocess

def run_command(cmd):
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

multi_prompt_variants = {
    'prototype_only_10x': {
        'prototype_source': 'manual',
        'prototype_json': PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_10x.json',
        'prompt_mode': 'prototype_only',
        'type_args': [],
        'max_epoch': '10',
        'scheduler_milestones': ['4', '8'],
    },
    'prototype_only_10x_v2': {
        'prototype_source': 'manual',
        'prototype_json': PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_10x_v2.json',
        'prompt_mode': 'prototype_only',
        'type_args': [],
        'max_epoch': '3',
        'scheduler_milestones': ['2'],
    },
    'prototype_only_5x_v3': {
        'prototype_source': 'manual',
        'prototype_json': PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_5x_v3.json',
        'prompt_mode': 'prototype_only',
        'type_args': [],
        'max_epoch': '3',
        'scheduler_milestones': ['2'],
    },
    'prototype_only_5x_v4': {
        'prototype_source': 'manual',
        'prototype_json': PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_5x_v4.json',
        'prompt_mode': 'prototype_only',
        'type_args': [],
        'max_epoch': '3',
        'scheduler_milestones': ['2'],
    },
}

# Default: run only the latest 5-prompt prototype-only v4 ablation so old runs are not repeated accidentally.
# To rerun older variants, add their names here, e.g. ['prototype_only_5x_v3', 'prototype_only_5x_v4'].
selected_multi_prompt_variants = ['prototype_only_5x_v4']

for variant_name in selected_multi_prompt_variants:
    cfg = multi_prompt_variants[variant_name]
    print('\n' + '=' * 80)
    print('Multi-prompt variant:', variant_name)

    epoch_dir = f'model/epoch_checkpoints_multi_prompt_{variant_name}_scratch'
    output_model = f'model/model_ucf_multi_prompt_{variant_name}_scratch.pth'
    checkpoint_path = f'model/checkpoint_multi_prompt_{variant_name}_scratch.pth'
    save_cur_path = f'model/model_cur_multi_prompt_{variant_name}_scratch.pth'
    diagnostics_dir = PROJECT_ROOT / 'code' / f'ucf_checkpoint_diagnostics_multi_prompt_{variant_name}_scratch'

    common_prompt_args = [
        '--prototype-source', cfg['prototype_source'],
        '--prototype-json', str(cfg['prototype_json']),
        '--prompt-mode', cfg['prompt_mode'],
        *cfg['type_args'],
    ]

    train_cmd = [
        'python', 'ucf_train_multi_prompt.py',
        '--feature-root', str(FEATURE_ROOT),
        *common_prompt_args,
        '--use-pretrained-model', 'false',
        '--train-list', '../list/ucf_CLIP_rgb_description.csv',
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
        '--lambda-consistency', '0.01',
        '--consistency-top-k-ratio', '0.15',
        '--regularize-score-branch', 'both',
        '--regularize-top-k-ratio', '0.15',
        '--lambda-normal-topk', '0.0',
        '--lambda-abnormal-bg', '0.0',
        '--lambda-ranking-margin', '0.0',
        '--ranking-margin', '0.20',
        '--lr', '2e-5',
        '--max-epoch', cfg['max_epoch'],
        '--scheduler-milestones', *cfg['scheduler_milestones'],
        '--eval-steps', '0',
        '--num-workers', '4',
        '--pin-memory', 'true',
        '--use-amp', 'false',
        '--epoch-checkpoint-dir', epoch_dir,
        '--output-model-path', output_model,
        '--checkpoint-path', checkpoint_path,
        '--save-cur-path', save_cur_path,
    ]
    print('Training:', ' '.join(shlex.quote(part) for part in train_cmd))
    run_command(train_cmd)

    eval_cmd = [
        'python', 'ucf_evaluate.py',
        '--feature-root', str(FEATURE_ROOT),
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--baseline-model-path', str(PRETRAINED_MODEL),
        '--description-model-path', output_model,
        '--description-model-type', 'multi_prompt',
        *common_prompt_args,
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
    ]
    print('Evaluating:', ' '.join(shlex.quote(part) for part in eval_cmd))
    run_command(eval_cmd)

    analyze_cmd = [
        'python', 'ucf_analyze_checkpoints.py',
        '--feature-root', str(FEATURE_ROOT),
        '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
        '--baseline-model-path', str(PRETRAINED_MODEL),
        '--epoch-checkpoint-dir', epoch_dir,
        '--description-model-path', output_model,
        '--finetuned-model-type', 'multi_prompt',
        *common_prompt_args,
        '--output-dir', str(diagnostics_dir),
        '--gt-path', '../list/gt_ucf.npy',
        '--gt-segment-path', '../list/gt_segment_ucf.npy',
        '--gt-label-path', '../list/gt_label_ucf.npy',
        '--timeline-count', '6',
    ]
    print('Analyzing:', ' '.join(shlex.quote(part) for part in analyze_cmd))
    run_command(analyze_cmd)


## 19. Weighted Multi-Prompt Training

Cell n?y train t? scratch v?i weighted prompt bank ???c t?o t? notebook analyze prompt. ??u v?o ch?nh l? `code/ucf_weighted_class_prompts.json`.

Kh?c v?i multi-prompt uniform, m?i class v?n c? 5 prompt nh?ng m?i prompt c? m?t tr?ng s? ri?ng. Prompt c? score semantic t?t h?n s? ?nh h??ng nhi?u h?n khi aggregate `prompt_logits [B,T,14,5]` th?nh `logits2 [B,T,14]`.

Trong l??t n?y b?t normal regularization ? m?c nh? b?ng `lambda-normal-topk = 0.05`. Hai regularizer c?n l?i l? `lambda-abnormal-bg` v? `lambda-ranking-margin` v?n gi? b?ng `0.0` ?? t?ch ri?ng t?c ??ng c?a normal suppression.


In [ ]:
import json
import shlex
import subprocess

if 'run_command' not in globals():
    def run_command(cmd):
        result = subprocess.run(cmd, text=True, capture_output=True)
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        result.check_returncode()

WEIGHTED_PROMPT_JSON = PROJECT_ROOT / 'code' / 'ucf_weighted_class_prompts.json'
if not WEIGHTED_PROMPT_JSON.exists():
    raise FileNotFoundError(
        f'Missing weighted prompt JSON: {WEIGHTED_PROMPT_JSON}\n'
        'Run analyze_vadclip_prompt_text_space.ipynb first to create this file.'
    )

with WEIGHTED_PROMPT_JSON.open('r', encoding='utf-8') as f:
    weighted_payload = json.load(f)

print('Weighted prompt JSON:', WEIGHTED_PROMPT_JSON)
print('Embedding space:', weighted_payload.get('metadata', {}).get('embedding_space'))
print('Weighting method:', weighted_payload.get('metadata', {}).get('weighting_method'))
print('Prompts per class:', weighted_payload.get('metadata', {}).get('prompts_per_class'))
print('Prompt weight temperature:', weighted_payload.get('metadata', {}).get('prompt_weight_temperature'))

print('\nPrompt weight preview:')
for class_name, weights in weighted_payload['prompt_weights'].items():
    prompts = weighted_payload['classes'][class_name]
    best_idx = max(range(len(weights)), key=lambda i: weights[i])
    worst_idx = min(range(len(weights)), key=lambda i: weights[i])
    print(
        f"{class_name:<13} best={weights[best_idx]:.3f} #{best_idx}: {prompts[best_idx]} | "
        f"lowest={weights[worst_idx]:.3f} #{worst_idx}: {prompts[worst_idx]}"
    )

variant_name = 'manual_caption_5x_v5_weighted_normal_reg'
epoch_dir = f'model/epoch_checkpoints_multi_prompt_{variant_name}_scratch'
output_model = f'model/model_ucf_multi_prompt_{variant_name}_scratch.pth'
checkpoint_path = f'model/checkpoint_multi_prompt_{variant_name}_scratch.pth'
save_cur_path = f'model/model_cur_multi_prompt_{variant_name}_scratch.pth'
diagnostics_dir = PROJECT_ROOT / 'code' / f'ucf_checkpoint_diagnostics_multi_prompt_{variant_name}_scratch'

common_prompt_args = [
    '--prototype-source', 'manual',
    '--prototype-json', str(WEIGHTED_PROMPT_JSON),
    '--prompt-weights-json', str(WEIGHTED_PROMPT_JSON),
    '--prompt-mode', 'manual_caption',
]

train_cmd = [
    'python', 'ucf_train_multi_prompt.py',
    '--feature-root', str(FEATURE_ROOT),
    *common_prompt_args,
    '--use-pretrained-model', 'false',
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lambda-consistency', '0.01',
    '--consistency-top-k-ratio', '0.15',
    '--regularize-score-branch', 'both',
    '--regularize-top-k-ratio', '0.15',
    '--lambda-normal-topk', '0.05',
    '--lambda-abnormal-bg', '0.0',
    '--lambda-ranking-margin', '0.0',
    '--ranking-margin', '0.20',
    '--lr', '2e-5',
    '--max-epoch', '3',
    '--scheduler-milestones', '2',
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--epoch-checkpoint-dir', epoch_dir,
    '--output-model-path', output_model,
    '--checkpoint-path', checkpoint_path,
    '--save-cur-path', save_cur_path,
]
print('\nTraining:', ' '.join(shlex.quote(part) for part in train_cmd))
run_command(train_cmd)

eval_cmd = [
    'python', 'ucf_evaluate.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--description-model-path', output_model,
    '--description-model-type', 'multi_prompt',
    *common_prompt_args,
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
]
print('\nEvaluating:', ' '.join(shlex.quote(part) for part in eval_cmd))
run_command(eval_cmd)

analyze_cmd = [
    'python', 'ucf_analyze_checkpoints.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--epoch-checkpoint-dir', epoch_dir,
    '--description-model-path', output_model,
    '--finetuned-model-type', 'multi_prompt',
    *common_prompt_args,
    '--output-dir', str(diagnostics_dir),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--timeline-count', '6',
]
print('\nAnalyzing:', ' '.join(shlex.quote(part) for part in analyze_cmd))
run_command(analyze_cmd)

print('\nSaved outputs:')
print('  model:', output_model)
print('  epoch checkpoints:', epoch_dir)
print('  diagnostics:', diagnostics_dir)


## 19. Training Với Prompt Đại Diện Được Chọn Tự Động

Cell này train từ scratch bằng bộ prompt đại diện được chọn từ notebook analyze. Mỗi class chỉ có 1 prompt. Chế độ hiện tại là `prototype_only`, nghĩa là prompt được đưa trực tiếp vào text encoder và không tự thêm tên class phía trước.

Hãy chạy cell này sau khi đã chạy `analyze_vadclip_prompt_text_space.ipynb` và file selected JSON đã tồn tại. Các regularizer normal/background/ranking cũng có thể bật trong cell này bằng cách đổi lambda.


In [ ]:
import shlex
import subprocess


def run_command(cmd):
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()


SELECTED_REPRESENTATIVE_PROMPT_JSON = PROJECT_ROOT / 'code' / 'ucf_selected_representative_class_prompts.json'
if not SELECTED_REPRESENTATIVE_PROMPT_JSON.exists():
    raise FileNotFoundError(
        f'Missing selected prompt JSON: {SELECTED_REPRESENTATIVE_PROMPT_JSON}\n'
        'Run code/analyze_vadclip_prompt_text_space.ipynb first to create it.'
    )

variant_name = 'prototype_only_selected_representative_1x'
epoch_dir = f'model/epoch_checkpoints_multi_prompt_{variant_name}_scratch'
output_model = f'model/model_ucf_multi_prompt_{variant_name}_scratch.pth'
checkpoint_path = f'model/checkpoint_multi_prompt_{variant_name}_scratch.pth'
save_cur_path = f'model/model_cur_multi_prompt_{variant_name}_scratch.pth'
diagnostics_dir = PROJECT_ROOT / 'code' / f'ucf_checkpoint_diagnostics_multi_prompt_{variant_name}_scratch'

common_prompt_args = [
    '--prototype-source', 'manual',
    '--prototype-json', str(SELECTED_REPRESENTATIVE_PROMPT_JSON),
    '--prompt-mode', 'prototype_only',
]

train_cmd = [
    'python', 'ucf_train_multi_prompt.py',
    '--feature-root', str(FEATURE_ROOT),
    *common_prompt_args,
    '--use-pretrained-model', 'false',
    '--train-list', '../list/ucf_CLIP_rgb_description.csv',
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--lambda-consistency', '0.01',
    '--consistency-top-k-ratio', '0.15',
    '--regularize-score-branch', 'both',
    '--regularize-top-k-ratio', '0.15',
    '--lambda-normal-topk', '0.0',
    '--lambda-abnormal-bg', '0.0',
    '--lambda-ranking-margin', '0.0',
    '--ranking-margin', '0.20',
    '--lr', '2e-5',
    '--max-epoch', '3',
    '--scheduler-milestones', '2',
    '--eval-steps', '0',
    '--num-workers', '4',
    '--pin-memory', 'true',
    '--use-amp', 'false',
    '--epoch-checkpoint-dir', epoch_dir,
    '--output-model-path', output_model,
    '--checkpoint-path', checkpoint_path,
    '--save-cur-path', save_cur_path,
]
print('\n' + '=' * 80)
print('Selected representative prompt variant:', variant_name)
print('Prompt JSON:', SELECTED_REPRESENTATIVE_PROMPT_JSON)
print('Training:', ' '.join(shlex.quote(part) for part in train_cmd))
run_command(train_cmd)

eval_cmd = [
    'python', 'ucf_evaluate.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--description-model-path', output_model,
    '--description-model-type', 'multi_prompt',
    *common_prompt_args,
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
]
print('Evaluating:', ' '.join(shlex.quote(part) for part in eval_cmd))
run_command(eval_cmd)

analyze_cmd = [
    'python', 'ucf_analyze_checkpoints.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--epoch-checkpoint-dir', epoch_dir,
    '--description-model-path', output_model,
    '--finetuned-model-type', 'multi_prompt',
    *common_prompt_args,
    '--output-dir', str(diagnostics_dir),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--timeline-count', '6',
]
print('Analyzing:', ' '.join(shlex.quote(part) for part in analyze_cmd))
run_command(analyze_cmd)

manual_prompt_diag_dir = PROJECT_ROOT / 'code' / f'ucf_manual_prompt_diagnostics_{variant_name}_scratch'
manual_prompt_diag_cmd = [
    'python', 'ucf_analyze_manual_prompts.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--manual-model-path', output_model,
    '--prompt-json', str(SELECTED_REPRESENTATIVE_PROMPT_JSON),
    '--prompt-mode', 'prototype_only',
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--output-dir', str(manual_prompt_diag_dir),
]
print('Manual prompt diagnostics:', ' '.join(shlex.quote(part) for part in manual_prompt_diag_cmd))
run_command(manual_prompt_diag_cmd)

print('\nSelected representative training outputs:')
print('  model:', output_model)
print('  epoch checkpoints:', epoch_dir)
print('  checkpoint diagnostics:', diagnostics_dir)
print('  prompt diagnostics:', manual_prompt_diag_dir)


## 20. Phân Tích Manual Prompt

Cell này không train model. Nó phân tích manual CLIP-friendly prompts để tìm class, video hoặc prompt nào gây tụt metric. Mặc định phân tích checkpoint được khai báo trong `MANUAL_MODEL_TO_ANALYZE`; nếu muốn phân tích epoch khác, chỉ cần đổi path checkpoint tương ứng.


In [ ]:
import shlex
import subprocess

def run_command(cmd):
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

MANUAL_PROMPT_JSON_TO_ANALYZE = PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_5x_v4.json'
MANUAL_MODEL_TO_ANALYZE = 'model/epoch_checkpoints_multi_prompt_manual_caption_5x_v4_scratch/model_epoch_002_multi_prompt.pth'
MANUAL_PROMPT_DIAGNOSTICS_DIR = PROJECT_ROOT / 'code' / 'ucf_manual_prompt_diagnostics_5x_v4'
# To analyze v3 epoch 2 instead:
# MANUAL_PROMPT_JSON_TO_ANALYZE = PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_5x_v3.json'
# MANUAL_MODEL_TO_ANALYZE = 'model/epoch_checkpoints_multi_prompt_manual_caption_5x_v3_scratch/model_epoch_002_multi_prompt.pth'
# MANUAL_PROMPT_DIAGNOSTICS_DIR = PROJECT_ROOT / 'code' / 'ucf_manual_prompt_diagnostics_5x_v3'
# To analyze v2 epoch 2 instead:
# MANUAL_PROMPT_JSON_TO_ANALYZE = PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_10x_v2.json'
# MANUAL_MODEL_TO_ANALYZE = 'model/epoch_checkpoints_multi_prompt_manual_caption_10x_v2_scratch/model_epoch_002_multi_prompt.pth'
# MANUAL_PROMPT_DIAGNOSTICS_DIR = PROJECT_ROOT / 'code' / 'ucf_manual_prompt_diagnostics_v2'
# To analyze v1 final instead:
# MANUAL_PROMPT_JSON_TO_ANALYZE = PROJECT_ROOT / 'code' / 'ucf_manual_class_prompts_10x.json'
# MANUAL_MODEL_TO_ANALYZE = 'model/model_ucf_multi_prompt_manual_caption_10x_scratch.pth'
# MANUAL_PROMPT_DIAGNOSTICS_DIR = PROJECT_ROOT / 'code' / 'ucf_manual_prompt_diagnostics'

manual_prompt_diag_cmd = [
    'python', 'ucf_analyze_manual_prompts.py',
    '--feature-root', str(FEATURE_ROOT),
    '--test-list', '../list/ucf_CLIP_rgbtest_relative.csv',
    '--baseline-model-path', str(PRETRAINED_MODEL),
    '--manual-model-path', MANUAL_MODEL_TO_ANALYZE,
    '--prompt-json', str(MANUAL_PROMPT_JSON_TO_ANALYZE),
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
    '--output-dir', str(MANUAL_PROMPT_DIAGNOSTICS_DIR),
]
print('Running:', ' '.join(shlex.quote(part) for part in manual_prompt_diag_cmd))
run_command(manual_prompt_diag_cmd)
